# Zero-TVM throughput bench — free Colab T4

**First: Runtime → Change runtime type → T4 GPU.**

Colab is CUDA-only by default, so cell 1 hand-creates the NVIDIA Vulkan ICD that Chrome's WebGPU needs. If it doesn't print **Tesla T4**, fall back to Vast.ai (there you can just set `NVIDIA_DRIVER_CAPABILITIES=all`).

In [ ]:
# 1) Node 22 + Chrome/Vulkan deps, then enable Vulkan on the T4
!nvidia-smi --query-gpu=name,driver_version --format=csv,noheader
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1
!apt-get -qq install -y nodejs vulkan-tools libvulkan1 xvfb libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libgbm1 libasound2 libxcomposite1 libxdamage1 libxrandr2 libxkbcommon0 libxfixes3 libdrm2 libpango-1.0-0 libcairo2 libatspi2.0-0 fonts-liberation > /dev/null
!mkdir -p /usr/share/vulkan/icd.d
!printf '{"file_format_version":"1.0.0","ICD":{"library_path":"libGLX_nvidia.so.0","api_version":"1.3.277"}}' > /usr/share/vulkan/icd.d/nvidia_icd.json
print('--- Vulkan device (must say Tesla T4 / NVIDIA): ---')
!vulkaninfo --summary 2>/dev/null | grep -E 'deviceName|driverName' || echo 'NO VULKAN DEVICE -> use Vast.ai instead'

In [ ]:
# 2) Clone + run the bench (first run downloads ~2 GB of Phi-3 weights)
!pip -q install "huggingface_hub[cli]"
!git clone -q https://github.com/abgnydn/zero-tvm.git
!cd zero-tvm && npm ci --silent && BENCH_HEADLESS=new bash bench/cloud-bench.sh

## Done

Copy the printed `results.json` into your repo at `bench/results.json`, then run `npm run bench:sync -- --write` locally (no GPU) to update BENCH.md + the bench page.